# Usage-head overfit sanity (separate from main training)

Samples 200 garments **with images on disk**, no aug / no class weights / dropout 0 / constant LR 3e-4 / 60 epochs.
Does not block `kaggle_train.ipynb`.

In [ ]:
import os
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

os.environ["PYTHONUNBUFFERED"] = "1"

def find_one(root: Path, name: str) -> Path:
    matches = [path for path in root.rglob(name) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError((name, matches))
    return matches[0]

probe = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    text=True,
).strip()
print(probe, flush=True)

input_root = Path("/kaggle/input")
package_root = Path("/kaggle/working/package")
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(find_one(input_root, "package.zip")) as archive:
    archive.extractall(package_root)
subprocess.check_call(
    [sys.executable, "-u", "-m", "pip", "install", "--no-deps", "--no-build-isolation", str(package_root)]
)

def image_layout(root: Path):
    if (root / "resized").is_dir():
        return root / "index.parquet", root / "resized"
    if (root / "station1").is_dir():
        return root / "index.parquet", root
    raise FileNotFoundError(root)

parquets = [path for path in input_root.rglob("index.parquet") if path.is_file()]
data_root = parquets[0].parent if len(parquets) == 1 else Path("/kaggle/working/data")
if not (data_root / "index.parquet").is_file():
    data_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(find_one(input_root, "bundle.zip")) as archive:
        archive.extractall(data_root)
index_path, image_root = image_layout(data_root)
print(data_root, image_root, flush=True)


In [ ]:
from garmentroute.train import train
from garmentroute.util import load_yaml
from garmentroute.views import find_subset_members

WORKING = Path("/kaggle/working")

def apply_paths(config: dict) -> dict:
    config["index"] = str(index_path)
    config["image_root"] = str(image_root)
    config["run_dir"] = str(WORKING / "runs")
    members = find_subset_members(package_root, package_root / "data", index_path.parent)
    if members is not None:
        config["subset_members"] = str(members)
    return config

cfg_path = package_root / "configs" / "usage_overfit.yaml"
cfg = apply_paths(load_yaml(cfg_path))
print("START usage_overfit", datetime.now(timezone.utc).isoformat(), flush=True)
train(cfg, cfg_path)
print("END usage_overfit", datetime.now(timezone.utc).isoformat(), flush=True)
summary = json.loads((WORKING / "runs" / "usage_overfit_200" / "train_summary.json").read_text())
scores = [row.get("train_usage_macro_f1") or 0.0 for row in summary.get("history", [])]
best = max(scores) if scores else 0.0
print("overfit best train_usage_macro_f1", best, flush=True)
if best < 0.8:
    print("WARNING: overfit did not reach 0.8 train macro-F1; investigate labels/loss.", flush=True)
else:
    print("overfit OK", flush=True)
